# Day 8 — Improve Extraction Using Regex and Phrase Matching

The Day 7 dictionary matcher does plain substring matching, which causes real false positives. This notebook demonstrates the bug on our own data, fixes it with regex word-boundaries, and adds spaCy phrase matching for multi-word skills.

## The bug, found in our own Day 7 output
The alias `ml` (for *Machine Learning*) is a **substring of `html`**, so the Day 7 extractor incorrectly tags every job mentioning HTML as also requiring Machine Learning.

In [2]:
print("'ml' in 'html' ->", 'ml' in 'html')

import pandas as pd
rb = pd.read_csv('../data/rule_based_skills.csv')
bad = rb[rb['job_title'] == 'Web Developer'].iloc[0]
print(bad['job_title'], '->', bad['rule_based_skills'])

'ml' in 'html' -> True
Web Developer -> ['CSS', 'Git', 'HTML', 'Java', 'JavaScript', 'Machine Learning', 'React']


## Fix: word-boundary regex matching
Using `\b...\b` ensures we only match whole words, and also lets us handle variants like `python3`, `python 3` in one pattern.

In [4]:
import re

df = pd.read_csv('../data/clean_jobs.csv')

regex_patterns = {
    'Python': r'\bpython(?:\s*3)?\b',
    'Machine Learning': r'\bmachine learning\b',
    'Deep Learning': r'\bdeep learning\b',
    'Natural Language Processing': r'\bnatural language processing\b',
    'Power BI': r'\bpower\s*bi\b',
    'Data Science': r'\bdata science\b',
    'SQL': r'\bsql\b',
    'HTML': r'\bhtml\b',
    'CSS': r'\bcss\b',
}

def regex_extract(text, patterns=regex_patterns):
    text = str(text)
    return sorted([name for name, pat in patterns.items() if re.search(pat, text)])

df['regex_skills'] = df['clean_description'].apply(regex_extract)

fixed = df[df['job_title'] == 'Web Developer'].iloc[0]
print('Fixed extraction (no more false Machine Learning):', fixed['regex_skills'])

Fixed extraction (no more false Machine Learning): ['CSS', 'HTML']


## Multi-word skills across the dataset (regex)

In [5]:
from collections import Counter
all_regex_skills = Counter()
for skills in df['regex_skills']:
    all_regex_skills.update(skills)
all_regex_skills.most_common()

[('SQL', 7472),
 ('Python', 4449),
 ('Power BI', 3498),
 ('Machine Learning', 1045),
 ('CSS', 976),
 ('HTML', 976)]

## Advanced task: spaCy phrase matching
`PhraseMatcher` matches exact multi-word phrases efficiently and is easy to extend with new skill phrases without writing new regex each time.

In [6]:
import spacy
from spacy.matcher import PhraseMatcher

nlp = spacy.load('en_core_web_sm', disable=['ner', 'parser'])
matcher = PhraseMatcher(nlp.vocab, attr='LOWER')

phrase_list = ['python', 'sql', 'power bi', 'machine learning', 'deep learning',
               'natural language processing', 'data science', 'docker', 'kubernetes',
               'aws', 'azure', 'react', 'javascript']
patterns = [nlp.make_doc(p) for p in phrase_list]
matcher.add('SKILLS', patterns)

def phrase_extract(text, matcher=matcher, nlp=nlp):
    doc = nlp(str(text))
    matches = matcher(doc)
    return sorted(set(doc[start:end].text for _, start, end in matches))

sample = df['clean_description'].iloc[0]
print(sample)
print('\nspaCy phrase matches:', phrase_extract(sample))

we are looking for a machine learning engineer to join our team at technova solutions in navi mumbai the candidate will analyze business data build reports collaborate with stakeholders and develop scalable solutions required skills include python machine learning tensorflow scikit learn docker experience with data quality documentation problem solving and communication is preferred the role requires 2 3 years of relevant experience

spaCy phrase matches: ['docker', 'machine learning', 'python']


## Apply phrase matching to a sample of the dataset (spaCy is slower, so we sample 300 rows)

In [7]:
sample_df = df.sample(300, random_state=1).copy()
sample_df['phrase_skills'] = sample_df['clean_description'].apply(phrase_extract)
sample_df[['job_title','phrase_skills']].head(10)

,job_title,phrase_skills
9953,Business Analyst,"[power bi, sql]"
3850,SQL Developer,[sql]
4962,Marketing Analyst,"[power bi, python, sql]"
3886,Product Analyst,"[python, sql]"
5437,HR Analyst,"[power bi, sql]"
8517,Software Engineer,"[python, sql]"
2041,Database Administrator,[sql]
1989,Software Engineer,"[python, sql]"
1933,HR Analyst,"[power bi, sql]"
9984,Machine Learning Engineer,"[docker, machine learning, python]"


## Summary
- **Bug found:** naive substring matching (Day 7) creates false positives, e.g. `ml` inside `html`
- **Fix 1 — regex with `\b` word boundaries:** eliminates the false positive and normalizes variants like `python`, `python 3`, `python3` into one canonical skill
- **Fix 2 — spaCy `PhraseMatcher`:** robust, fast, easy-to-extend multi-word phrase matching

**Deliverable:** `advanced_skill_extractor.ipynb` (this notebook).